# ML パイプライン自動化 v2（リネージ対応）

## 概要

`03_Automation.ipynb`（v1）では SPCS Service + 生POSテーブルを使用していたため、
Snowflake のリネージ（データ系統追跡）に **Feature Store → Model → 出力テーブル** の
関係が記録されませんでした。

本ノートブックでは以下の2点を修正し、リネージを完全に追跡できるようにします。

### v1 → v2 の変更点

| 項目 | v1 (03_Automation) | v2 (本ノートブック) |
|------|-------------------|-------------------|
| **特徴量ソース** | 生POSテーブル（CTE内で特徴量再構築） | `SALES_FORECAST_FV$v2`（Feature View の Dynamic Table） |
| **推論呼び出し** | `SALES_FORECAST_SERVICE!PREDICT()` (SPCS) | `SALES_FORECAST_MODEL!PREDICT()` (Model Registry) |
| **リネージ** | FV・Model との紐付けなし | FV → Model → RESULTS が Snowsight で追跡可能 |

### なぜリネージが途切れていたか

1. **Feature View との断絶**: 生テーブル (ID_POS_TRANSACTIONS, PRODUCT_MASTER) から
   CTE で特徴量を再構築していたため、Feature View `SALES_FORECAST_FV$v2` の
   Dynamic Table を経由しておらず、Snowflake がデータフローを追跡できなかった
2. **Model との断絶**: SPCS Service (`SALES_FORECAST_SERVICE!PREDICT()`) を
   経由していたため、Model Registry のモデルとの直接的な呼び出し関係が記録されなかった

### 修正後のリネージ

```
BUYER_AGENT.ID_POS_TRANSACTIONS  ─┐
BUYER_AGENT.PRODUCT_MASTER        ─┤
                                   ↓
    SALES_ML."SALES_FORECAST_FV$v2" (Feature View Dynamic Table)
                                   ↓
    SALES_ML.SALES_FORECAST_MODEL!PREDICT() (Model Registry)
                                   ↓
    SALES_ML.SALES_FORECAST_RESULTS (出力テーブル)
```

### 前提条件
- `FOODEX_DEMO.SALES_ML` スキーマ作成済み
- Feature View `SALES_FORECAST_FV` (v2) 登録済み
- Model `SALES_FORECAST_MODEL` が Registry 登録済み（`TargetPlatform.WAREHOUSE`）
- Model Monitor `SALES_FORECAST_MONITOR` 設定済み

## Step 1: 環境セットアップ

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()
session.sql("USE DATABASE FOODEX_DEMO").collect()
session.sql("USE SCHEMA SALES_ML").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()

print("Database:  FOODEX_DEMO")
print("Schema:    SALES_ML")
print("Warehouse: COMPUTE_WH")

## Step 2: 依存オブジェクトの確認

v2 では以下の2つが必須です:
1. **Feature View** `SALES_FORECAST_FV$v2` — 特徴量の読み取り元
2. **Model** `SALES_FORECAST_MODEL` — SQL推論の呼び出し先（`TargetPlatform.WAREHOUSE`）

In [ ]:
print("=== Feature View (Dynamic Table) ===")
session.sql("""
    SELECT TABLE_NAME, SCHEDULING_STATE, TARGET_LAG, DATA_TIMESTAMP
    FROM INFORMATION_SCHEMA.DYNAMIC_TABLES
    WHERE TABLE_SCHEMA = 'SALES_ML'
      AND TABLE_NAME LIKE 'SALES_FORECAST_FV%'
""").show()

print("\n=== Feature View のカラム ===")
session.sql('SELECT * FROM FOODEX_DEMO.SALES_ML."SALES_FORECAST_FV$v2" LIMIT 3').show()

print("\n=== Models ===")
session.sql("SHOW MODELS IN SCHEMA FOODEX_DEMO.SALES_ML").show()

print("\n=== Model Monitors ===")
session.sql("SHOW MODEL MONITORS IN SCHEMA FOODEX_DEMO.SALES_ML").show()

## Step 3: 予測結果テーブルの作成

Task が INSERT する先のテーブルです（v1 と同じ定義）。

| カラム | 型 | 説明 |
|--------|------|------|
| FORECAST_DATE | DATE | 予測対象日（今日+1〜+7） |
| CATEGORY_MEDIUM | VARCHAR | 商品カテゴリ |
| PREDICTED_SALES | FLOAT | 予測売上額 |
| CREATED_AT | TIMESTAMP_NTZ | 予測実行日時 |

In [ ]:
CREATE TABLE IF NOT EXISTS FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS (
    FORECAST_DATE    DATE            NOT NULL,
    CATEGORY_MEDIUM  VARCHAR(100)    NOT NULL,
    PREDICTED_SALES  FLOAT           NOT NULL,
    CREATED_AT       TIMESTAMP_NTZ   NOT NULL DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = '日次7日間売上予測結果（DAILY_SALES_FORECAST_TASK で毎朝自動INSERT）';

## Step 4: 推論SQLの動作確認（リネージ対応版）

### v1 との違い

| 処理 | v1 | v2 |
|------|----|----|  
| 特徴量取得 | 生POSテーブルからCTEで再構築（6段CTE） | `SALES_FORECAST_FV$v2` から直接SELECT（1段CTE） |
| 推論呼び出し | `SALES_FORECAST_SERVICE!PREDICT()` | `SALES_FORECAST_MODEL!PREDICT()` |
| CTE段数 | 5段（daily_agg → base_features → latest_features → category_encoding → forecast_days） | 3段（fv_latest → category_encoding → forecast_days） |

### v2 の処理フロー
1. **fv_latest**: Feature View DT から最新日の特徴量を取得（LAG/MA は計算済み）
2. **category_encoding**: DENSE_RANK でカテゴリをエンコーディング
3. **forecast_days**: 7日分のオフセットを生成
4. **PREDICT**: Model Registry のモデルで推論実行

In [ ]:
WITH fv_latest AS (
    SELECT
        CATEGORY_MEDIUM,
        LAG_1, LAG_2, LAG_3, LAG_7, LAG_14, MA_7,
        TXN_COUNT, TOTAL_QTY,
        SALES_DATE
    FROM FOODEX_DEMO.SALES_ML."SALES_FORECAST_FV$v2"
    WHERE SALES_DATE = (
        SELECT MAX(SALES_DATE)
        FROM FOODEX_DEMO.SALES_ML."SALES_FORECAST_FV$v2"
    )
),
category_encoding AS (
    SELECT DISTINCT
        CATEGORY_MEDIUM,
        DENSE_RANK() OVER (ORDER BY CATEGORY_MEDIUM) - 1 AS CATEGORY_ENCODED
    FROM FOODEX_DEMO.SALES_ML."SALES_FORECAST_FV$v2"
),
forecast_days AS (
    SELECT SEQ4() + 1 AS DAY_OFFSET
    FROM TABLE(GENERATOR(ROWCOUNT => 7))
)
SELECT
    DATEADD('day', d.DAY_OFFSET, CURRENT_DATE()) AS FORECAST_DATE,
    f.CATEGORY_MEDIUM,
    FOODEX_DEMO.SALES_ML.SALES_FORECAST_MODEL!PREDICT(
        f.LAG_1, f.LAG_2, f.LAG_3, f.LAG_7, f.LAG_14, f.MA_7,
        DAYOFWEEK(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        CASE WHEN DAYOFWEEK(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())) IN (0, 6) THEN 1 ELSE 0 END,
        MONTH(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        QUARTER(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        f.TXN_COUNT, f.TOTAL_QTY, c.CATEGORY_ENCODED
    ):PREDICTED_SALES::FLOAT AS PREDICTED_SALES,
    CURRENT_TIMESTAMP() AS CREATED_AT
FROM fv_latest f
CROSS JOIN forecast_days d
JOIN category_encoding c ON f.CATEGORY_MEDIUM = c.CATEGORY_MEDIUM
WHERE f.LAG_14 IS NOT NULL
ORDER BY f.CATEGORY_MEDIUM, FORECAST_DATE;

## Step 5: Snowflake Task の作成（リネージ対応版）

### Task チェーン構成
```
DAILY_SALES_FORECAST_TASK (Root, 毎朝AM6時 JST)
  │  SALES_FORECAST_FV$v2 → 特徴量取得
  │  → SALES_FORECAST_MODEL!PREDICT (Model Registry)
  │  → SALES_FORECAST_RESULTS に7日間予測をINSERT
  ↓
DAILY_DRIFT_CHECK_TASK (Child, 推論完了後)
     Model Monitor の PSI をチェック
```

### リネージが追跡される理由
- Task が `SALES_FORECAST_FV$v2`（Feature View の裏テーブル）を **直接 SELECT** → FV → RESULTS のリネージが記録される
- Task が `SALES_FORECAST_MODEL!PREDICT()` を **SQL関数として呼び出し** → Model → RESULTS のリネージが記録される

In [ ]:
CREATE OR REPLACE TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = 'USING CRON 0 6 * * * Asia/Tokyo'
    COMMENT = '毎朝6時に7日間の売上予測を実行（Feature View + Model Registry 経由、リネージ対応）'
AS
INSERT INTO FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS (FORECAST_DATE, CATEGORY_MEDIUM, PREDICTED_SALES, CREATED_AT)
WITH fv_latest AS (
    SELECT
        CATEGORY_MEDIUM,
        LAG_1, LAG_2, LAG_3, LAG_7, LAG_14, MA_7,
        TXN_COUNT, TOTAL_QTY,
        SALES_DATE
    FROM FOODEX_DEMO.SALES_ML."SALES_FORECAST_FV$v2"
    WHERE SALES_DATE = (
        SELECT MAX(SALES_DATE)
        FROM FOODEX_DEMO.SALES_ML."SALES_FORECAST_FV$v2"
    )
),
category_encoding AS (
    SELECT DISTINCT
        CATEGORY_MEDIUM,
        DENSE_RANK() OVER (ORDER BY CATEGORY_MEDIUM) - 1 AS CATEGORY_ENCODED
    FROM FOODEX_DEMO.SALES_ML."SALES_FORECAST_FV$v2"
),
forecast_days AS (
    SELECT SEQ4() + 1 AS DAY_OFFSET
    FROM TABLE(GENERATOR(ROWCOUNT => 7))
)
SELECT
    DATEADD('day', d.DAY_OFFSET, CURRENT_DATE()) AS FORECAST_DATE,
    f.CATEGORY_MEDIUM,
    FOODEX_DEMO.SALES_ML.SALES_FORECAST_MODEL!PREDICT(
        f.LAG_1, f.LAG_2, f.LAG_3, f.LAG_7, f.LAG_14, f.MA_7,
        DAYOFWEEK(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        CASE WHEN DAYOFWEEK(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())) IN (0, 6) THEN 1 ELSE 0 END,
        MONTH(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        QUARTER(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        f.TXN_COUNT, f.TOTAL_QTY, c.CATEGORY_ENCODED
    ):PREDICTED_SALES::FLOAT AS PREDICTED_SALES,
    CURRENT_TIMESTAMP() AS CREATED_AT
FROM fv_latest f
CROSS JOIN forecast_days d
JOIN category_encoding c ON f.CATEGORY_MEDIUM = c.CATEGORY_MEDIUM
WHERE f.LAG_14 IS NOT NULL;

### ドリフト検知 Child Task

推論完了後に Model Monitor の PSI を確認し、ドリフトを検知します。

In [ ]:
CREATE OR REPLACE PROCEDURE FOODEX_DEMO.SALES_ML.SP_CHECK_MODEL_DRIFT()
    RETURNS VARCHAR
    LANGUAGE SQL
AS
$$
DECLARE
    drift_count INTEGER;
    max_psi FLOAT;
    result_msg VARCHAR;
BEGIN
    SELECT COUNT(*), COALESCE(MAX(METRIC_VALUE), 0)
    INTO :drift_count, :max_psi
    FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
        'SALES_FORECAST_MONITOR',
        'PSI',
        'LAG_1',
        'DAY',
        DATEADD('day', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
    ))
    WHERE METRIC_VALUE >= 0.25;

    IF (:drift_count > 0) THEN
        result_msg := 'ALERT: Critical drift detected! PSI=' || :max_psi::VARCHAR || ' (' || :drift_count::VARCHAR || ' days). Consider model retraining.';
    ELSE
        SELECT COALESCE(MAX(METRIC_VALUE), 0) INTO :max_psi
        FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            'SALES_FORECAST_MONITOR',
            'PSI',
            'LAG_1',
            'DAY',
            DATEADD('day', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ));
        result_msg := 'OK: No critical drift. Max PSI=' || :max_psi::VARCHAR;
    END IF;

    RETURN result_msg;
END;
$$;

In [ ]:
CREATE OR REPLACE TASK FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK
    USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
    COMMENT = '推論完了後にModel Monitorのドリフト検知を実行'
    AFTER FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK
AS
    CALL FOODEX_DEMO.SALES_ML.SP_CHECK_MODEL_DRIFT();

## Step 6: Task の有効化

Task チェーンでは **Child → Root** の順に RESUME する必要があります。

**注意**: 有効化するとスケジュールに従って自動実行が開始されます。

In [ ]:
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK RESUME;

In [ ]:
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK RESUME;

In [ ]:
print("=== Task 一覧 ===")
session.sql("SHOW TASKS IN SCHEMA FOODEX_DEMO.SALES_ML").show()

## Step 7: 手動実行テスト

`EXECUTE TASK` でスケジュールを待たずに即時実行できます。
Root Task を実行すると、完了後に Child Task (ドリフト検知) も自動実行されます。

In [ ]:
EXECUTE TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK;

In [ ]:
import time

print("Task 実行状況を確認中...")
for i in range(12):
    time.sleep(10)
    result = session.sql("""
        SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, ERROR_MESSAGE
        FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
            TASK_NAME => 'DAILY_SALES_FORECAST_TASK',
            SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP())
        ))
        ORDER BY SCHEDULED_TIME DESC
        LIMIT 1
    """).to_pandas()

    if len(result) > 0:
        state = result.iloc[0]['STATE']
        print(f"  [{i+1}/12] State: {state}")
        if state == 'SUCCEEDED':
            print(f"\nTask completed successfully!")
            break
        elif state == 'FAILED':
            print(f"\nTask failed!")
            print(f"  Error: {result.iloc[0]['ERROR_MESSAGE']}")
            break
    else:
        print(f"  [{i+1}/12] Waiting...")
else:
    print("\nTask still running. Check history later.")

## Step 8: 予測結果の確認

In [ ]:
SELECT *
FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
WHERE CREATED_AT = (SELECT MAX(CREATED_AT) FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS)
ORDER BY CATEGORY_MEDIUM, FORECAST_DATE;

In [ ]:
print("=== 予測結果サマリー ===")
session.sql("""
    SELECT
        CREATED_AT::DATE AS BATCH_DATE,
        COUNT(*) AS TOTAL_RECORDS,
        COUNT(DISTINCT CATEGORY_MEDIUM) AS CATEGORIES,
        MIN(FORECAST_DATE) AS FORECAST_FROM,
        MAX(FORECAST_DATE) AS FORECAST_TO,
        ROUND(SUM(PREDICTED_SALES), 0) AS TOTAL_PREDICTED_SALES
    FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
    GROUP BY BATCH_DATE
    ORDER BY BATCH_DATE DESC
    LIMIT 7
""").show()

In [ ]:
import matplotlib.pyplot as plt

latest_forecast = session.sql("""
    SELECT FORECAST_DATE, CATEGORY_MEDIUM, PREDICTED_SALES
    FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
    WHERE CREATED_AT = (SELECT MAX(CREATED_AT) FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS)
    ORDER BY CATEGORY_MEDIUM, FORECAST_DATE
""").to_pandas()

if len(latest_forecast) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    categories = latest_forecast['CATEGORY_MEDIUM'].unique()
    for cat in categories:
        cat_data = latest_forecast[latest_forecast['CATEGORY_MEDIUM'] == cat]
        axes[0].plot(cat_data['FORECAST_DATE'], cat_data['PREDICTED_SALES'],
                     marker='o', label=cat, linewidth=2)
    axes[0].set_xlabel('Forecast Date')
    axes[0].set_ylabel('Predicted Sales (円)')
    axes[0].set_title('7日間売上予測（カテゴリ別）')
    axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    axes[0].grid(alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)

    cat_totals = latest_forecast.groupby('CATEGORY_MEDIUM')['PREDICTED_SALES'].sum().sort_values(ascending=True)
    axes[1].barh(cat_totals.index, cat_totals.values, color='steelblue')
    axes[1].set_xlabel('7日間合計予測売上 (円)')
    axes[1].set_title('カテゴリ別 7日間合計予測')
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("予測結果がINSERTされるとグラフが表示されます。")

## Step 9: リネージの確認

修正により、以下のリネージが Snowsight 上で確認できるようになります。

### 確認方法
1. Snowsight → **Data** → `FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS` を選択
2. **Lineage** タブを開く
3. 以下のフローが表示されるはず:

```
ID_POS_TRANSACTIONS ─┐
PRODUCT_MASTER      ─┤
                     ↓
  SALES_FORECAST_FV$v2 (Feature View DT)
                     ↓
  SALES_FORECAST_MODEL (Model Registry)
                     ↓
  SALES_FORECAST_RESULTS
```

### プログラムでの確認

`OBJECT_DEPENDENCIES` 関数でリネージを SQL で確認できます。

In [ ]:
SELECT
    REFERENCING_OBJECT_NAME AS SOURCE_OBJECT,
    REFERENCING_OBJECT_DOMAIN AS SOURCE_TYPE,
    REFERENCED_OBJECT_NAME AS TARGET_OBJECT,
    REFERENCED_OBJECT_DOMAIN AS TARGET_TYPE
FROM TABLE(INFORMATION_SCHEMA.OBJECT_DEPENDENCIES(
    OBJECT_NAME => 'FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK',
    OBJECT_TYPE => 'TASK'
))
ORDER BY SOURCE_OBJECT;

In [ ]:
SELECT
    REFERENCING_OBJECT_NAME AS UPSTREAM_OBJECT,
    REFERENCING_OBJECT_DOMAIN AS UPSTREAM_TYPE,
    REFERENCED_OBJECT_NAME AS DOWNSTREAM_OBJECT,
    REFERENCED_OBJECT_DOMAIN AS DOWNSTREAM_TYPE
FROM TABLE(INFORMATION_SCHEMA.OBJECT_DEPENDENCIES(
    OBJECT_NAME => 'FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS',
    OBJECT_TYPE => 'TABLE'
))
ORDER BY UPSTREAM_OBJECT;

## Step 10: Task 実行履歴の確認

In [ ]:
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME,
       DATEDIFF('second', QUERY_START_TIME, COMPLETED_TIME) AS DURATION_SEC,
       ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    TASK_NAME => 'DAILY_SALES_FORECAST_TASK',
    SCHEDULED_TIME_RANGE_START => DATEADD('day', -7, CURRENT_TIMESTAMP())
))
ORDER BY SCHEDULED_TIME DESC
LIMIT 10;

In [ ]:
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME,
       RETURN_VALUE, ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    TASK_NAME => 'DAILY_DRIFT_CHECK_TASK',
    SCHEDULED_TIME_RANGE_START => DATEADD('day', -7, CURRENT_TIMESTAMP())
))
ORDER BY SCHEDULED_TIME DESC
LIMIT 10;

## Step 11: 古い予測結果のクリーンアップ（オプション）

過去の予測結果が蓄積されるため、定期的にクリーンアップする Task も設定できます。

In [ ]:
CREATE OR REPLACE TASK FOODEX_DEMO.SALES_ML.WEEKLY_FORECAST_CLEANUP_TASK
    USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
    SCHEDULE = 'USING CRON 0 0 * * 0 Asia/Tokyo'
    COMMENT = '毎週日曜深夜に30日以上前の予測結果を削除'
AS
    DELETE FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
    WHERE CREATED_AT < DATEADD('day', -30, CURRENT_TIMESTAMP());

In [ ]:
-- 必要に応じて有効化
-- ALTER TASK FOODEX_DEMO.SALES_ML.WEEKLY_FORECAST_CLEANUP_TASK RESUME;

## 運用コマンド集

### Task の一時停止（Root → Child の順）
```sql
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK SUSPEND;
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK SUSPEND;
```

### Task の再開（Child → Root の順）
```sql
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK RESUME;
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK RESUME;
```

### Task の即時実行
```sql
EXECUTE TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK;
```

### 最新の予測結果を確認
```sql
SELECT * FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
WHERE CREATED_AT = (SELECT MAX(CREATED_AT) FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS)
ORDER BY CATEGORY_MEDIUM, FORECAST_DATE;
```

### リネージの確認
```sql
-- SALES_FORECAST_RESULTS の上流オブジェクトを確認
SELECT * FROM TABLE(INFORMATION_SCHEMA.OBJECT_DEPENDENCIES(
    OBJECT_NAME => 'FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS',
    OBJECT_TYPE => 'TABLE'
));
```

### Task の削除
```sql
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK SUSPEND;
DROP TASK IF EXISTS FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK;
DROP TASK IF EXISTS FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK;
DROP TASK IF EXISTS FOODEX_DEMO.SALES_ML.WEEKLY_FORECAST_CLEANUP_TASK;
```

## まとめ

### v1 → v2 変更一覧

| 変更箇所 | v1 (03_Automation) | v2 (本ノートブック) | 効果 |
|----------|-------------------|-------------------|------|
| 特徴量ソース | `ID_POS_TRANSACTIONS` + `PRODUCT_MASTER` | `"SALES_FORECAST_FV$v2"` | Feature View → RESULTS リネージ |
| 推論呼び出し | `SALES_FORECAST_SERVICE!PREDICT()` | `SALES_FORECAST_MODEL!PREDICT()` | Model → RESULTS リネージ |
| CTE段数 | 5段（特徴量を毎回再構築） | 3段（FVから直接取得） | SQL簡潔化 + パフォーマンス向上 |
| SPCS依存 | 必須（Service が READY でないと失敗） | 不要（Model Registry の WH 推論） | 運用簡素化 |

### 作成したオブジェクト

| オブジェクト | 名前 | 説明 |
|------------|------|------|
| Table | SALES_FORECAST_RESULTS | 7日間売上予測結果テーブル |
| Stored Procedure | SP_CHECK_MODEL_DRIFT | ドリフト検知（PSI閾値チェック） |
| Task (Root) | DAILY_SALES_FORECAST_TASK | 毎朝6時(JST) FV → Model推論 → INSERT |
| Task (Child) | DAILY_DRIFT_CHECK_TASK | 推論完了後にドリフト検知 |
| Task (独立) | WEEKLY_FORECAST_CLEANUP_TASK | 毎週日曜 古い予測結果を削除 |

### リネージ（Snowsight で確認可能）

```
BUYER_AGENT.ID_POS_TRANSACTIONS ─┐
BUYER_AGENT.PRODUCT_MASTER      ─┤
                                 ↓
  SALES_ML."SALES_FORECAST_FV$v2"  (Feature View Dynamic Table)
                                 ↓
  SALES_ML.SALES_FORECAST_MODEL    (Model Registry, WAREHOUSE推論)
                                 ↓
  SALES_ML.SALES_FORECAST_RESULTS  (出力テーブル)
```

In [ ]:
print("=" * 60)
print("  ML Pipeline Automation v2 (Lineage-Enabled)")
print("=" * 60)
print(f"\n[推論方式]")
print(f"  Feature View DT + Model Registry WAREHOUSE推論")
print(f"  SPCS不要、純SQL、リネージ完全対応")
print(f"\n[リネージ]")
print(f"  SALES_FORECAST_FV$v2 → SALES_FORECAST_MODEL → SALES_FORECAST_RESULTS")
print(f"  Snowsight の Lineage タブで確認可能")
print(f"\n[Task チェーン]")
print(f"  DAILY_SALES_FORECAST_TASK : 毎朝6時(JST) [Root]")
print(f"    → FV$v2 から最新特徴量取得")
print(f"    → MODEL!PREDICT で推論")
print(f"    → SALES_FORECAST_RESULTS に7日間予測INSERT")
print(f"  DAILY_DRIFT_CHECK_TASK    : 推論完了後    [Child]")
print(f"    → Model Monitor PSI チェック")
print(f"\n[結果テーブル]")
print(f"  FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS")
print(f"\n[全オブジェクト所在]")
print(f"  スキーマ: FOODEX_DEMO.SALES_ML")